   ... expected to know how to use CBuild/`cb`, plus normal C tools like `gcc`,
   `gdb` and `valgrind`. The PDF also says compilation/crashing penalties can
   be harsh, so clean compiling code matters a lot.

   Source context...

---
KEY CONCEPT
   Raw compiler command:
   
```c
gcc -std=c17 -Wall -Wpedantic -o prog prog.c
```
   means: manually tell GCC flags, output names, and source files.

   CBuild/`cb` is closer to:

```c
cb
```
   meaning: read the provided build config, work out what to compile/link, then
   call the compiler for you.

   So mentally:
```
gcc/clang = direct compiler command
make      = build automation using Makefile rules
cb        = course build automation using provided .cbuild/.build config
```
   You do not need to write pattern rules. You mainly need to know:
```bash
cb
cb clean
```
   and understand that GCC warnings/errors still matter because `cb` eventually
   invokes GCC/Clang underneath. 


---

   Use the `wc` command to count the number of lines, words, and bytes in the 
   files specified by the File parameter. If a file is not specified for the
   File parameter, standard input is used.

```sh
cb
cb --clean
cb --allclean
cb --test
cb --install
```

   Locally `cb` is not on this machine's PATH... 

---
CONCEPTS
   CBuild/`cb` is basically "Course Make without writing Makefiles."

   In `08.cbuild/test1/.cbuild`, the whole config is:
```make
BUILD = avgwordlen testlist
```
   That means: by default, build two executable programs, `avgwordlen` and
   `testlist`.

   `cb` scans the ``.c` and `.h` files, especially local includes like:

```c
#include "intlist.h"
```
   Then it infers dependencies:

```
testlist.c includes intlist.h
intlist.h has matching intlist.c
therefore testlist needs testlist.o + intlist.o
```
   The raw GCC equivalent would be:
```sh
gcc -Wall -c intlist.c
gcc -Wall -c testlist.c
gcc testlist.o intlist.o -o testlist
```

   ... But wth `cb`, you just do:
```sh
cb
```

   Important distinction:
```
compiler error = bad C syntax/types in one source file
linker error   = object files compiled, but final executable cannot be connected
runtime crash  = program built, then segfaulted/asserted/etc
```

...
```c
intlist_append(xs, 3);
```
   Linker error if `intlist_append` is declared in `.h` but not implemented in 
   `.c`.
```
int *p = NULL;
*p = 3;
```
   Runtime crash..

   For clean builds:
```sh
cb --clean
```
   removes generated objects/executables.

```sh
cb --allclean
```
   cleans and rebuilds.

   This matters in exams because stale object files can hide problems, and the
   hints PDF says non-compiling/crashing tasks get heavy penalties.

---

Q1

```sh
cb
```

---
Q2

   Conceptual hint: `avgwordlen.c` and `testlist.c` are both main programs.

   Syntax hints: multiple `main` functions cannot be linked into one executable.

   ANSWER: It tries to compile/link every `.c` file into one program. If more
   than one file has `main`, linking fails with a multiple-definition error.


---
Q3

   Manually, what two-stage GCC process builds `testlist` from `testlist.c`
   and `intlist.c`?

   Conceptual hint: first compile `.c -> .o`, then link `.o -> executable`.

   Syntax hint: use `-c` for compile-only.

```sh
gcc -Wall -c testlist.c
gcc -Wall -Wextra -c intlist.c
gcc testlist.o intlist.o -o testlist
```

---
Q4
   `testlist.c` includes `intlist.h`. There is also an `intlist.c`. What does
   `cb` infer?

   CONCEPTUAL HINT: matching `.h` and `.c` form a module.

   Answer: `testlist` depends on the `intlist` module, so `cb` should compile 
   and link `intlist.o` into `testlist`.

---
Q5
   ... implementation changed, not interface.

   ANSWER: `intlist.o` should recompile, and programs using it, such as 
   `testlist` and `avgwordlen`, should relink.

---
Q6
   ... interface changed. ... Anything including `intlist.h` should recompile,
   so likely `intlist.o`, `testlist.o`, `avgwordlen.o`, then the executables
   relink.

---
Q7
```sh
undefined reference to `intlist_length`
```
   ... This is link error...

   ANSWER: Linker error. The function was declared or called, but no matching
   implementation was found during linking. 

---
Q8
```
warning: implicit declaration of function `foo`
```
   CONCEPTUAL HINT: Compiler has not seen the function prototype.

   ANSWER: Missing header include, misspelled function name, or no declaration
   before use. In C17 with strict flags, treat this as serious.

   warning: implicit declaration of function `foo`

---
Q9
   In current `cb` syntax, what command forces a clean rebuild?

```sh
cb --allclean
```

---
Q10
   ... Your solution to one task has broken helper function that stops 
   compilation. What should you do.

   ANSWER: Comment out or isolate the broken code so the task compiles. A 
   smaller compiling solution is better than a larger non-compiling one. 

---

   This specific compiler error normally happens when a developer is 
   transitioning from small, single-file hobby scripts to MULTI-FILE SOFTWARE
   ARCHITECTURES. In a simple project, running a single command like 
   `gcc main.c -o program` works because the compiler handles both compilation
   and linking in one seamless pass. However, when a second like `intlist.c` is 
   introduced, the natural but mistaken instinct is to just copy-paste the new
   filename into the middle of the old command and 

---

   In C, a string is not a first-class object type; it is simply a contiguous
   array of characters in memory that concludes with a special marker called the
   NULL TERMINATOR (`\0`). Because there are no built-in properties like `.len`
   or `.length`, finding the length of a string requires scanning through
   memory byte-by-byte until this null terminator is encountered.

1. The Standard Way: `strlen()`
   The canonical method to find the length of a `const char*` is using the 
   `strlen()` function provided by the standard library header `<string.h>`.

```c
#include <stdio.h>
#include <string.h>

int main() {
    const char *str = "ARMv8";

    // strlen returns a size_t (unsigned integer type)
    size_t len = strlen(str);

    printf("The length is: %zu\n", len);    // Outputs: 5
    return 0;
}
```
   - EXCLUDES THE NULL TERMINATOR: `strlen()` counts only the readable 
     characters. For `"ARMv8"`, it returns `5`, even though the array occupies
     `6` bytes of memory in total to store the trailing `\0`.
   - TIME COMPLEXITY: This operation is O(N). Because C strings do not store
     their length complexity. `strlen()` must inspect every single character
     sequentially. Calling `strlen()` inside the condition of a loop 
     (e.g., `for (size_t i = 0; i < strlen(s); i++)`) creates an accidental
     O(N^2) performance bottleneck.


---
2. THE DEFENSIVE WAY: `strnlen()`
   If you are dealing with untrusted input or raw binary packets where a string 
   might be malformed or missing its null terminator, `strlen()` will read
   past the buffer boundaries until it crashes with a segmentation fault.

   To prevent this, use `strnlen()`. It requires a secondary argument specifying
   the maximum number of bytes to examine. 

```c
#include <string.h>

const char *uncrusted_buffer = get_network_packet();
// Stops scanning at the null terminator OR after 100 bytes, whichever comes
// first
size_t len = strnlen(untrusted_buffer, 100);
```


---
3. THE MANUAL WAY (Under the Hood)
   To understand exactly how the machine calculates this value without library
   functions, you can implemnent the counting logic manually using a pointer or 
   an index tracker:

```c
size_t manual_strlen(const char *s) {
    if (s == NULL) {
        return NULL;
    }

    size_t length = 0;
    while (s[length] != '\0') {
        length++;
    }
    return length;
}
```

   - `strlen(ptr)` computes the number of characters up to the null terminator
     by evaluating the memory contents at runtime.
   - `sizeof(ptr)` computes the size of the pointer variable itself at compile
     time. On a 64-bit architecture, a pointer is always 8 bytes, regardless of
     whether the string it points to contains 2 characters or 20,000 characters.


---

---

   ... arrays, pointer iteration, `sizeof`, command-line args, `getchar`, 
   `fgets`, and pointer-output parameters are the main refresh targets.

CORE CONCEPTS
   Array decay to pointers when passed to functions, so this works in `main`.

```c
sizeof(arr) / sizeof(arr[0])
```
   but not inside:

```c
void f(int arr[]) { ... }
```
   because `arr` is now basically `int *`.
   
   Pointer iteration uses the "one past the end" idiom:

```
int *end = arr + n;
for (int *p = arr; p < endl; p++) {
    total += *p;
}
```

---

   Character input should use `int`, not `char`, because `EOF` may not fit in 
   `char`:

```c
int c;
while ((c = getchar()) != EOF) { ... }
```
   
   Line input uses `fgets`, which keeps the newline if it fits:
```c
char buf[100];
while (fgets(buf, sizeof buf, stdin) != NULL) { ... }
```


   Command-line args:
```c
int main(int argc, char **argv) {
   
}
```

---

CORE CONCEPTS: `getchar()` and `fgets()`
   Before diving into the questions... navigating standard input in C.

1. The `getchar()` Engine
   `getchar()` reads exactly ONE SINGLE CHARACTER at a time from standard
   input (`stdin`).
   - THE `int` RETURN TRAP: `getchar()` does not return an `int`. This is 
     because you it needs to be able to return every possible character value 
     plus a special architectural signal called `EOF` (End of File, typically
     defined as `-1`). If you ... can cause a bug where `EOF` is misread as a 
     valid character.
   - BUFFERING: Input is line-buffered. If a user types `ABC` and hits Enter,
     `getchar()` does not fire immediately when they type `A`. It waits for 
     the Enter key, then reads `A`, then `B`, then `C`, and finally the newline
     character `'\n'`.

2. The `fgets()` Buffer
   `fgets()` reads an ENTIRE LINE OF TEXT safely into a memory buffer.
   - SIGNATURE: `char *fgets(char *str, int num, FILE *stream);`
   - THE NEWLINE PRESERVATION: If there is enough room in your buffer, `fgets`
     will read the text AND INCLUDE the TRAILING CHARACTER `'\n'` right inside
     your string before adding the null terminator `'\0'`. This means if a 
     use inputs "hello", the string in memory becomes `"hello\n\0"`.


---
QUESTION 1: Character Counting with `getchar()` (Easy)
   TASK: Write a function `int count_spaces(void)` that reads characters from 
   the terminal one by one using `getchar()`. Count and return the total number
   of spaces (`' '`) the user types. Stop readiin the moment the user hits Enter
   (a newline `'\n'`) or if `EOF` is reached.

CONCEPTUAL HINTS
   - You need a loop that calls `getchar()` on every single iteration.
   - Inside the loop, check if the character matches a space. If it does, 
     increment a counter.
   - The loop's termination condition must check for both the newline character
     and `EOF`.

```c
#include <stdio.h>

int count_spaces(void) {
    int space_count = 0;
    int c = getchar(stdin);
    while (c != EOF && c != '\n') {
        if (c == ' ') {
            space_count++;
        }
        c = getchar(stdin);
    }
    return space_count;
}
```
   EXPLANATION: The syntax `(c = getchar() != '\n')` performs two actions 
   simultaneously: it assigns the incoming character to `c`, and then 
   immediately evaluates whether `c` is a newline. Because `c` is declared as an
   `int`, it safely evaluates against `EOF` without risk of truncation. 


---
QUESTION 2: Clean Line Trimming with `fgets()` (Medium)

```c
#include <stdio.h>
#include <string.h>

size_t read_and_trim(char *buffer, int max_size) {
    char *str = fgets(buffer, max_size, stdin);
    if (str == NULL) {
        return 0;
    }
    char *end = str + strlen(str);
    for (char *ptr = str; ptr != end; ptr++) {
        if (*ptr == '\n') {
            *ptr = '\0';
            break;
        }
    }
    return strlen(str);
}
```

```c
#include <stdio.h>
#include <string.h>

size_t read_and_trim(char *buffer, int max_size) {
    // 1. Read the raw line safely
    if (fgets(buffer, max_size, stdin) == NULL) {
        return 0;
    }

    // 2. Traversal using pointer arithmetic to strip '\n'
    char *p = buffer;
    while (*p != '\0') {
        if (*p == '\n') {
            *p = '\0';      // Overwrite newline with null terminator
            break;          // Stop scanning
        }
        p++;                // Advance to the next character element
    }

    // 3. Return the clean string length
    return strlen(buffer);
}
```
   EXPLANATION: `fgets` pulls the text along with the trailing `'\n'`. By 
   setting up `char *p = buffer`, `p` points directly to the first character.
   The loop executed `p++`... When it finds `'\n'`, it overwrites that memory
   address with `'\0'`, shortening the string and stripping the line break
   cleanly.


---
Q3: Capitalize and Count Words
   TASK: Write a function `int process_input_line(void)` that reads a string up
   to 100 characters long using `fgets()`.

```c
#include <stdio.h>
#include <string.h>

int process_input_line(void) {
    char buffer[100];
    int in_word = 0;
    size_t word_count = 0;

    // Read up to 100 characters into our buffer
    if (fgets(buffer, sizeof(buffer), stdin) == NULL) {
        return 0;
    }

    size_t max_len = 100;
    size_t len = strnlen(buffer, max_len);
    if (len == (max_len - 1) && buffer[max_len - 2] != '\n') {
        fprintf(stderr, "ERROR: Inputted line exceeds 100 characters.");
        return -1;
    }

    char* end = buffer + len;
    for (char* ptr = buffer; ptr != end; ptr++) {
        if (*ptr >= 'a' && *ptr <= 'z') {
            *ptr = (char)(*ptr - 32);   // Convert to uppercase via ASCII offset
        }

        if (*ptr == ' ' || *ptr == '\n') {
            in_word = 0;
        } else if (in_word == 0) {
            in_word = 1;
            word_count++;
        }
    }

    printf("MODIFIED LINE; %s", buffer);

    return (int)word_count;
}
```





---

A1
---
```c
#include <stdio.h>

int main(int argc, char **argv) {
    for (int i = 0; i < argc; i++) {
        printf("argv[%d] = %s\n", i, argv[i]);
    }
    return 0;
}
```


A2
---
```c
#include <stdlib.h>

int sum_array(const int arr[], int n) {
    size_t total = 0;
    for (int i = 0; i < n; i++) {
        total += arr[i];
    }
    return (int)total;
}
```


A3
---
int sum_array_ptr(const int arr[], int n) {
    const int *end = arr + n;
    int total = 0;
    for (const int *ptr = arr; ptr < end; ptr++) {
        total += *ptr;
    }
    return total;
}


A4
---
   `arr` has decayed to a pointer. `sizeof(arr)` gives the size of a pointer, 
   not the number of bytes in the original array. Pass the length separately.


A5
---
```c
#include <stdio.h>

int main(int argc, char **argv) {
    int c_in = getchar();
    while (c_in != EOF) {
        putchar(cin);
        cin = getchar();
    }
    return 0;
}
```


A6
---
```c
#include <stdio.h>
#include <string.h>

#define BUFFER_SIZE 100

int main(int argc, char** argv) {
    char buffer[BUFFER_SIZE];
    
    while(fgets(buffer, sizeof(buffer), stdin) != NULL) {
        size_t len = strnlen(buffer, BUFFER_SIZE);

        if (len > 0 && bug[len - 1] == '\n') {
            buf[len - 1] = '\0';
        }

        printf("%s\n", buf);
    }

    return 0;
}
```


A7
---
If input ends before a newline, `getchar()` returns `EOF` forever, so the loop
can become infinite. Safer:

```c
int main() {
    int c = getchar();
    while (c != '\n' && c != EOF) {
        c = getchar();
    }
    return 0;
}
```


A8
---
```c
void divmod(int a, int b, int *div_out, int *mod_out) {
    *div_out = a / b;
    *mod_out = a % b;
}
```


A9
---
   `x` is a local stack variable. It dies when `bad` returns, so returning `&x`
   creates a dangling pointer.


A10
---
```c
int main(int argc, char** argv) {
    if (argc != 2) {
        fprintf(stderr, "ERROR: Main expects 1 argument only\n");
        return -1;
    }

    char *end = NULL;
    long n = strtol(argv[1], &end, 10);
    
    if (*end != '\0') {
        fprintf(stderr, "bad integer: %s\n", argv[1]);
        return 1;
    }

    printf("%ld\n", n);
    return 0;
}
```







---

   In C, `const int *ptr` means THE UNDERLYING VALUE is READ-ONLY, but the 
   POINTER'S ADDRESS CAN CHANGE FREELY.

   To never get confused by `const` in pointers again, use the golden rule of C:
   READ POINTER DELCARATIONS from RIGHT TO LEFT (backwards).

---

   When you invoke 